# Lang-Bridge bench — GPU speed (E1) and video identity (E2)

Measures, before we build (plan: `docs/SCALE_PLAN.md`, Phase A):

- **E1**: how fast separation, Whisper and OmniVoice run with each setting, how full the
  GPU gets, and whether faster settings change the result; separation from an Opus copy.
- **E2**: whether the same video is recognised across copies, re-downloads, formats,
  trims and added intros, and never confused with other episodes.

Runs on Colab, Kaggle or your own GPU machine (see below). **Run all** (about an hour on a
T4). Paste the last cell's output back.

## Where it runs

| | Before **Run all** |
|---|---|
| **Kaggle** | Import this notebook (File → Import notebook → Link: `https://raw.githubusercontent.com/LetPastorSpeakInYourLanguage/lang_recomp/main/notebooks/lang_bridge.ipynb`); Settings → Accelerator **GPU T4 x2** (both are used), Internet **on** (phone-verified account). Use `STORAGE = bucket` to keep results. |
| **Colab** | Runtime → Change runtime type → **T4 GPU**. Drive is connected by itself when a setting points into `/content/drive/…`. |
| **Your own machine** | Jupyter with Python ≥ 3.10 (a fresh virtual environment is best). An NVIDIA GPU is used if there is one, otherwise the CPU (fine for transcripts, slow for voices). PyTorch and ffmpeg are installed if missing. |

Everything is set in the **Settings** cell, the token and bucket included.

In [ ]:
#@title Settings
SOURCES = "https://www.youtube.com/watch?v=xwseWCSXD3Y https://www.youtube.com/watch?v=SC_opiKLohg https://www.youtube.com/watch?v=D9jZMLm72a8"  #@param {type:"string"}
#@markdown E1 videos: links or file paths, separated by spaces (add a sermon from your Drive if you like).
SECONDS = 120  #@param {type:"integer"}
#@markdown Length of each excerpt for E1.
E2_MAIN = "https://www.youtube.com/watch?v=xwseWCSXD3Y https://www.youtube.com/watch?v=SC_opiKLohg"  #@param {type:"string"}
E2_OTHERS = "https://www.youtube.com/watch?v=MSJMJxd1udk https://www.youtube.com/watch?v=hb1CBEENiPQ https://www.youtube.com/watch?v=vxoPApiNZBU https://www.youtube.com/watch?v=-idY8F7LOSE https://www.youtube.com/watch?v=m7IlyBEyi3c"  #@param {type:"string"}
#@markdown E2: videos to recognise, and other videos that must never match them.
RUN_SEPARATION = True  #@param {type:"boolean"}
RUN_WHISPER = True  #@param {type:"boolean"}
RUN_OMNIVOICE = True  #@param {type:"boolean"}
RUN_IDENTITY = True  #@param {type:"boolean"}

HF_TOKEN = ""  #@param {type:"string"}
#@markdown Your [Hugging Face token](https://huggingface.co/settings/tokens) (write access if you use a bucket). **Never share or commit this notebook with a token filled in.**
DEVICE = "auto"  #@param ["auto", "cpu", "1 GPU", "2 GPUs"]
#@markdown `auto`: every GPU there is (Kaggle T4 x2 → 2, Colab T4 → 1), else the CPU.
WORK_DIR = ""  #@param {type:"string"}
#@markdown Code, models and scratch files. Empty: `/kaggle/working`, `/content`, or `~/lang-bridge` on your own machine.
CODE_BRANCH = "main"  #@param {type:"string"}

In [ ]:
#@title Get Lang-Bridge
import importlib.util, io, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

def _runtime():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    try:
        if importlib.util.find_spec('google.colab'):
            return 'colab'
    except (ImportError, ValueError):
        pass
    return 'local'

RUNTIME = _runtime()
ROOT = Path(WORK_DIR).expanduser() if WORK_DIR.strip() else Path({'kaggle': '/kaggle/working', 'colab': '/content'}.get(RUNTIME) or Path.home() / 'lang-bridge')
ROOT.mkdir(parents=True, exist_ok=True)
BRANCH = CODE_BRANCH.strip() or 'main'
# On your own machine, a notebook opened from inside a Lang-Bridge checkout uses that checkout.
_here = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'worker' / 'lb_worker').is_dir() and (p / 'app').is_dir()), None)
if RUNTIME == 'local' and _here:
    CODE = _here  # as it is, on whatever branch it has checked out (CODE_BRANCH is not used)
    BRANCH = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'checkout'
else:
    CODE = ROOT / 'lang_recomp'
    if shutil.which('git'):
        if not (CODE / '.git').exists():
            shutil.rmtree(CODE, ignore_errors=True)
            subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, 'https://github.com/LetPastorSpeakInYourLanguage/lang_recomp', str(CODE)], check=True)
        else:
            subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', '--depth', '1', 'origin', BRANCH], check=True)
            subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
    else:  # no git on this machine: the branch as a zip
        shutil.rmtree(CODE, ignore_errors=True)
        with urllib.request.urlopen('https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/archive/refs/heads/' + BRANCH + '.zip') as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(ROOT)
        next(p for p in ROOT.glob('lang_recomp-*') if p.is_dir()).rename(CODE)
sys.path[:0] = [str(CODE), str(CODE / 'worker')]
_v = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h (%cd)'], capture_output=True, text=True).stdout.strip() if (CODE / '.git').exists() else 'zip'
print(f'Lang-Bridge {BRANCH} {_v} in {CODE} · running on {RUNTIME}')

In [ ]:
#@title Set up this machine
from lb_worker import env
M = env.setup(RUNTIME, ROOT, DEVICE, 'folder', str(ROOT / 'lb-bench'), '', HF_TOKEN)
OUT = str(M['workspace'])  # results: lb-bench in WORK_DIR
os.environ.setdefault('LB_CACHE', OUT + '/cache')

In [ ]:
#@title E1 · separation settings and the Opus copy
from pathlib import Path
from lb_worker.bench import gpu
out = Path(OUT)
items = gpu.prepare(SOURCES.split(), out, seconds=SECONDS)
if RUN_SEPARATION:
    sep = gpu.bench_separation(items, out)
    copy = gpu.bench_audio_copy(items, out, sep)

In [ ]:
#@title E1 · Whisper batch size
if RUN_WHISPER:
    whisper = gpu.bench_whisper(items, out)

In [ ]:
#@title E1 · OmniVoice takes per call
import json
if RUN_OMNIVOICE:
    sep_dir = Path(next(r['dir'] for r in json.loads((out / 'e1_separation.json').read_text())['rows'] if r.get('dir')))
    ref, ref_text = gpu.reference(items, sep_dir, out)
    omni = gpu.bench_omnivoice(ref, ref_text, out)

In [ ]:
#@title E2 · video identity
from lb_worker.bench import identity
from lb_worker.deps import ensure
from lb_worker.stages.bulk import fetch
if RUN_IDENTITY:
    ensure('yt-dlp', probe='yt_dlp')
    d = out / 'identity'
    get = lambda url, name, h: fetch({'id': name, 'url': url}, d / f'{name}_{h}', h)
    others = [get(u, f'o{k}', 360) for k, u in enumerate(E2_OTHERS.split())]
    mains = []
    for k, u in enumerate(E2_MAIN.split()):
        vid = u.split('v=')[-1][:11]
        mains.append({'name': f'm{k}', 'video': get(u, f'm{k}', 720), 'redownload': get(u, f'm{k}', 360),
                      'other': others[0], 'links': identity.link_forms(vid) if 'youtube' in u else []})
    ident = identity.run(mains, others, out)

In [ ]:
#@title Summary (paste this back)
import json
summary = {}
for name in ('e1_separation', 'e1_audio_copy', 'e1_whisper', 'e1_omnivoice', 'e2_identity'):
    f = out / f'{name}.json'
    if f.exists():
        r = json.loads(f.read_text())
        rows = [{k: v for k, v in x.items() if k != 'dir'} for x in r.get('rows', [])]
        summary[name] = {k: v for k, v in r.items() if k not in ('rows', 'links')} | {'rows': rows}
print(json.dumps(summary, ensure_ascii=False))